# Embedding Space Exploration

Visualise the full set of precomputed reaction embeddings in 2-D using PCA and UMAP, coloured by EC enzyme class.

> Requires `uv sync --extra all` and precomputed embeddings in `data/embeddings/` (see README's Data pipeline and Training sections to regenerate them, or copy an existing run).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

## Load embeddings and EC labels

In [ ]:
EMB_PATH    = "data/embeddings/medium/embeddings.npy"
SMARTS_PATH = "data/embeddings/medium/smarts.txt"
RAW_FILES   = [
    "data/raw/retrorules-v3.0-metanetx.csv",
    "data/raw/retrorules-v3.0-rhea.csv",
]

embeddings   = np.load(EMB_PATH)                              # (N, d_model)
smarts_list  = Path(SMARTS_PATH).read_text().splitlines()

print(f"Loaded {embeddings.shape[0]:,} embeddings  dim={embeddings.shape[1]}")

In [ ]:
# join EC labels from raw RetroRules CSVs
raw = pd.concat([pd.read_csv(f) for f in RAW_FILES], ignore_index=True)
raw = raw[["TEMPLATE", "ECS"]].dropna(subset=["TEMPLATE"]).drop_duplicates("TEMPLATE")
raw["ec1"] = raw["ECS"].str.split(";").str[0].str.split(".").str[0]  # top-level class

df = pd.DataFrame({"smarts": smarts_list})
df = df.merge(raw[["TEMPLATE", "ec1"]].rename(columns={"TEMPLATE": "smarts"}),
              on="smarts", how="left")
df["ec1"] = df["ec1"].fillna("unknown")

print(df["ec1"].value_counts().to_string())

## PCA — quick 2-D overview

PCA is fast on the full dataset and gives an initial feel for the global structure.

In [ ]:
EC_COLORS = {
    "1": "#2563eb",  # Oxidoreductases
    "2": "#dc2626",  # Transferases
    "3": "#16a34a",  # Hydrolases
    "4": "#d97706",  # Lyases
    "5": "#7c3aed",  # Isomerases
    "6": "#0891b2",  # Ligases
    "7": "#be185d",  # Translocases
    "unknown": "#d1d5db",
}
EC_NAMES = {
    "1": "Oxidoreductases", "2": "Transferases", "3": "Hydrolases",
    "4": "Lyases",          "5": "Isomerases",  "6": "Ligases",
    "7": "Translocases",    "unknown": "Unknown",
}

colors = df["ec1"].map(EC_COLORS).fillna("#d1d5db").values

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)
print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}")

In [ ]:
# subsample for legibility
rng = np.random.default_rng(0)
idx = rng.choice(len(coords), size=min(20_000, len(coords)), replace=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(coords[idx, 0], coords[idx, 1],
           c=colors[idx], s=2, alpha=0.4, linewidths=0, rasterized=True)

handles = [
    mpatches.Patch(color=EC_COLORS[k], label=f"EC {k} — {EC_NAMES[k]}")
    for k in sorted(EC_COLORS) if k != "unknown"
]
ax.legend(handles=handles, fontsize=7, markerscale=3, loc="upper right")
ax.set_xlabel(f"PC 1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC 2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA of reaction embeddings (coloured by EC class)")
plt.tight_layout()
plt.show()

## UMAP — non-linear 2-D projection

UMAP preserves local neighbourhood structure better than PCA. Runs on a subsample to keep it interactive; save `umap_coords.npy` to reuse without recomputing.

In [ ]:
import umap

UMAP_CACHE = Path("data/embeddings/umap_coords.npy")
N_UMAP     = 50_000    # increase for more coverage (slower)

rng = np.random.default_rng(1)
umap_idx = rng.choice(len(embeddings), size=min(N_UMAP, len(embeddings)), replace=False)

if UMAP_CACHE.exists():
    umap_coords = np.load(UMAP_CACHE)
    print(f"Loaded cached UMAP coords from {UMAP_CACHE}")
else:
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                        metric="cosine", random_state=42, verbose=True)
    umap_coords = reducer.fit_transform(embeddings[umap_idx])
    np.save(UMAP_CACHE, umap_coords)
    print(f"Saved UMAP coords to {UMAP_CACHE}")

umap_colors = df["ec1"].iloc[umap_idx].map(EC_COLORS).fillna("#d1d5db").values

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
           c=umap_colors, s=1.5, alpha=0.4, linewidths=0, rasterized=True)

ax.legend(handles=handles, fontsize=7, markerscale=4, loc="upper right")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(f"UMAP of reaction embeddings — {len(umap_idx):,} reactions")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Embedding statistics

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(norms, bins=80, color="steelblue", edgecolor="none")
axes[0].set_xlabel("L2 norm")
axes[0].set_ylabel("count")
axes[0].set_title("Embedding norms")

dim_vars = embeddings.var(axis=0)
axes[1].plot(np.sort(dim_vars)[::-1], color="steelblue", lw=1)
axes[1].set_xlabel("dimension (sorted)")
axes[1].set_ylabel("variance")
axes[1].set_title("Per-dimension variance")

plt.tight_layout()
plt.show()

print(f"norm  mean={norms.mean():.2f}  std={norms.std():.2f}  min={norms.min():.2f}  max={norms.max():.2f}")